# Pneumonia detection — Colab pipeline

Runs the full training and evaluation pipeline on a Colab GPU (T4 / V100 / A100).

## What this notebook does
1. Clone the repo + install dependencies
2. Authenticate with Kaggle and download the dataset
3. (Optional) Mount Google Drive for persistent checkpoint storage
4. Train the 5-fold ResNet50 ensemble at 288×288 (~25 minutes on T4)
5. Evaluate the ensemble on the held-out test set
6. Render the standard set of report figures

## Hardware
Make sure runtime is GPU. *Runtime → Change runtime type → T4 GPU* (or better).

## Time estimate
- Setup + dataset download: ~3 min
- 5-fold ResNet50 @ 288 (15 epochs each, early stopping): ~25 min on T4
- Eval + plots: ~3 min
- **Total: ~30 min on a free T4**

(Same pipeline takes ~7 hours on AMD Vega 64 + DirectML on Windows.)

## 1. Clone repo + verify GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
%cd /content
!git clone https://github.com/Cyberwookie69/Pneumonia-xray-training.git
%cd /content/Pneumonia-xray-training

## 2. Install dependencies

Colab already has PyTorch with CUDA. We just need the project's extras.

In [ ]:
!pip install -q timm grad-cam opencv-python-headless kaggle

## 3. Kaggle authentication

Upload your `kaggle.json` (download from https://www.kaggle.com/settings → "Create New API Token").

In [ ]:
from google.colab import files
uploaded = files.upload()  # select kaggle.json
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

## 4. Download dataset (~2.3 GB, ~1 min)

In [ ]:
!python pneumonia.py

## 5. (Optional) Persist checkpoints to Google Drive

Colab sessions die after ~12 h and lose `/content/`. Mount Drive to keep your checkpoints.
Skip this cell if you don't care about persistence.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/pneumonia_runs', exist_ok=True)
os.environ['PNEUMONIA_RUNS'] = '/content/drive/MyDrive/pneumonia_runs'
print(f"Runs will be saved to: {os.environ['PNEUMONIA_RUNS']}")

## 6. Train 5-fold ResNet50 @ 288 (~25 min on T4)

This trains 5 separate models (one per fold), each with ImageNet-pretrained ResNet50, focal loss, and 15 epochs of fine-tuning with early stopping.

In [ ]:
!python pneumonia_run_folds.py --pretrained --extra="--img_size 288 --num_workers 4"

## 7. (Optional) Train ConvNeXt-Tiny ensemble + SNR-AdamW ablation

In [ ]:
# ConvNeXt-Tiny @ 224 — ~25 min on T4
!python pneumonia_run_folds.py --pretrained --extra="--model convnext_tiny.fb_in22k_ft_in1k" --tag cnx224

# SNR-AdamW ablation (Litman & Guo 2026) — ~15 min on T4
!python pneumonia_run_folds.py --pretrained --extra="--snr_optimizer" --tag snr_r50

## 8. Per-architecture eval + multi-arch ensemble

In [ ]:
!python pneumonia_eval.py --ensemble ens_f0,ens_f1,ens_f2,ens_f3,ens_f4 --use_best --num_workers 0
# If you also ran cnx224 + snr_r50:
# !python pneumonia_eval.py --ensemble cnx224_f0,cnx224_f1,cnx224_f2,cnx224_f3,cnx224_f4 --use_best --num_workers 0
# !python pneumonia_eval.py --ensemble snr_r50_f0,snr_r50_f1,snr_r50_f2,snr_r50_f3,snr_r50_f4 --use_best --num_workers 0
# Multi-arch (15 models):
# !python pneumonia_eval.py --ensemble ens_f0,ens_f1,ens_f2,ens_f3,ens_f4,cnx224_f0,cnx224_f1,cnx224_f2,cnx224_f3,cnx224_f4,snr_r50_f0,snr_r50_f1,snr_r50_f2,snr_r50_f3,snr_r50_f4 --use_best --num_workers 0

## 9. Plots

In [ ]:
!python pneumonia_plots.py curves --runs ens_f0,ens_f1,ens_f2,ens_f3,ens_f4
!python pneumonia_plots.py features --run ens_f0 --use_best
!python pneumonia_gradcam.py --run_name ens_f0 --use_best --n_samples 8

## 10. Display the figures inline

In [ ]:
from IPython.display import Image, display
import os

runs_root = os.environ.get('PNEUMONIA_RUNS', 'runs')
for relpath in [
    f'{runs_root}/plots/learning_curves_ens.png',
    f'{runs_root}/plots/fold_best_val_ens.png',
    f'{runs_root}/ens_f0/plots/tsne_best.png',
]:
    if os.path.exists(relpath):
        print(relpath)
        display(Image(relpath))